In [ ]:
import os
import google.generativeai as genai
from typing import Optional, Dict
from tools.match_song import match_song_by_mood
from tools.load_memory import MoodMemory
from kaggle_secrets import UserSecretsClient

try:
    GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    genai.configure(api_key=GOOGLE_API_KEY)
    print("✅ Gemini API key setup complete.")
except Exception as e:
    print(
        f"🔑 Authentication Error: Please make sure you have added 'GOOGLE_API_KEY' to your Kaggle secrets. Details: {e}"
    )

# Create the model
gemini = genai.GenerativeModel("models/gemini-2.5-flash")


In [ ]:
# Mood memory to track recent moods
mood_memory = MoodMemory(k=3)

In [ ]:
def infer_mood_from_text(user_input: str) -> str:
    prompt = f"""
You're an emotional assistant. A user just said:
"{user_input}"

From this, extract the user's mood using just one lowercase word (e.g., happy, sad, romantic, calm, energetic). 
Don't explain. Just return the single mood word.
"""
    try:
        response = model.generate_content(prompt)
        return response.text.strip().lower()
    except Exception as e:
        print("NotebookLM failed to infer mood. Error:", e)
        return "calm"  # default fallback

def mood_recommendation_agent(user_message: str, session_id: str = "default") -> Dict[str, str]:
    # Step 1: Detect mood (Notebook-style if input is long)
    user_mood = (
        infer_mood_from_text(user_message)
        if len(user_message.strip().split()) > 2 else user_message.strip().lower()
    )

    # Step 2: Log it in memory
    mood_memory.update(session_id, user_mood)

    # Step 3: Get music suggestion
    recommendation = match_song_by_mood(user_mood)

    # Step 4: Compose Gemini response
    mood_history = mood_memory.get_memory(session_id)
    prompt = f"""
You are a mood-matching assistant. Based on the user's current mood and recent patterns, craft a warm, empathetic, engaging message that introduces the recommended playlist or song.

Current mood: {user_mood}
Recent moods: {mood_history}

Music suggestion:
{recommendation}

Only write the user-facing message. Use friendly, supportive language. If the user has been sad for a while, show some care. If they’re consistently energetic, cheer them on.
"""
    try:
        response = model.generate_content(prompt)
        final_reply = response.text
    except Exception as e:
        print("Gemini Error:", e)
        final_reply = f"Gemini failed. Basic recommendation:\n{recommendation}"

    return {
        "reply": final_reply,
        "mood": user_mood,
        "history": mood_history,
        "source": recommendation.get("source", "N/A")
    }